# VEC-IoT MASAC — Revised Complete Colab

No nearest-edge baseline. Bitrate is network-only; load is actual per-edge queued processing work.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT='/content/drive/MyDrive/vec_iot_project'
os.chdir(PROJECT)
print(os.getcwd())


In [ ]:
!pip -q install -U numpy scipy scikit-learn matplotlib pandas pillow torch
req=os.path.join(PROJECT,'requirements_phase3.txt')
if os.path.exists(req):
    !pip -q install -r "$req"


In [ ]:
!python -m py_compile src/*.py train_masac.py evaluate_masac.py
print('Compilation successful.')


## Network diagnostic


In [ ]:
!python diagnose_network.py --vehicles datasets/vehicles.xml --tasks datasets/tasks.xml --max-tasks 500


## Quick training — run this first


In [ ]:
!python train_masac.py --vehicles datasets/vehicles.xml --tasks datasets/tasks.xml --episodes 3 --max-tasks 1000


## Full training — rerun from scratch with the revised files


In [ ]:
!python train_masac.py --vehicles datasets/vehicles.xml --tasks datasets/tasks.xml --episodes 40 --max-tasks 15000


## Evaluation


In [ ]:
!python evaluate_masac.py --vehicles datasets/vehicles.xml --tasks datasets/tasks.xml --model outputs/masac.pt --max-tasks 10000


# Load results


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
OUT=os.path.join(PROJECT,'outputs')
evaluation=pd.read_csv(os.path.join(OUT,'evaluation.csv'))
scenarios=pd.read_csv(os.path.join(OUT,'evaluation_scenarios.csv'))
edge_load=pd.read_csv(os.path.join(OUT,'evaluation_edge_load.csv'))
uav_positions=pd.read_csv(os.path.join(OUT,'uav_positions.csv'))
uav_presence=pd.read_csv(os.path.join(OUT,'uav_presence.csv'))
print('routes:')
print(evaluation['route'].value_counts())
display(evaluation.head())


## Average total latency: UAV relay vs no UAV relay


In [ ]:
uav=scenarios[scenarios['scenario']=='UAV-relay']
no_uav=scenarios[scenarios['scenario']=='No-UAV-relay']
vals=pd.Series({
    'UAV Relay': uav['latency_s'].mean() if len(uav) else np.nan,
    'No UAV Relay': no_uav['latency_s'].mean() if len(no_uav) else np.nan
})
print(vals)
ax=vals.plot(kind='bar')
ax.set_title('Average Total Latency: UAV Relay vs No UAV Relay')
ax.set_xlabel('Scenario'); ax.set_ylabel('Total latency (s)')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Average total energy


In [ ]:
v=evaluation['energy_j'].mean()
print(f'Average total energy: {v:.6f} J')
ax=pd.Series({'MASAC':v}).plot(kind='bar'); ax.set_title('Average Total Energy'); ax.set_ylabel('Energy (J)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Packet loss — network routes only


In [ ]:
network=evaluation[np.isfinite(pd.to_numeric(evaluation['avg_rate_bps'],errors='coerce'))].copy()
if len(network):
    v=network['packet_loss'].mean(); print(f'Average network packet loss: {v:.6f}')
    ax=pd.Series({'MASAC network routes':v}).plot(kind='bar'); ax.set_title('Average Packet Loss'); ax.set_ylabel('Packet loss'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else:
    print('No network/offloaded MASAC tasks were selected.')


## Average data rate — network routes only


In [ ]:
network=evaluation[np.isfinite(pd.to_numeric(evaluation['avg_rate_bps'],errors='coerce'))].copy()
if len(network):
    v=network['avg_rate_bps'].mean()/1e6; print(f'Average network data rate: {v:.3f} Mbps')
    ax=pd.Series({'MASAC network routes':v}).plot(kind='bar'); ax.set_title('Average Network Data Rate'); ax.set_ylabel('Mbps'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else:
    print('No network/offloaded MASAC tasks were selected.')


## Load distribution among edge servers


In [ ]:
load_by_edge=edge_load.groupby('edge')['load'].mean()
print(load_by_edge)
ax=load_by_edge.plot(kind='bar'); ax.set_title('Load Distribution Among Edge Servers'); ax.set_xlabel('Edge server'); ax.set_ylabel('Average queued processing work (s)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Task success rate


In [ ]:
rate=(evaluation['deadline_miss_s']<=1e-12).mean()*100
print(f'Task success rate: {rate:.2f}%')
ax=pd.Series({'MASAC':rate}).plot(kind='bar'); ax.set_title('Task Success Rate'); ax.set_ylabel('Success rate (%)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Average UAV presence


In [ ]:
v=uav_presence['uav_count'].mean() if len(uav_presence) else 0.0
print(f'Average UAV presence: {v:.3f} UAVs/timestep')


## UAV movement on the map


In [ ]:
plt.figure(figsize=(9,7))
for uid,g in uav_positions.groupby('uav_id'):
    plt.plot(g['x'],g['y'],marker='o',markersize=2,label=f'UAV {uid}')
plt.title('UAV Movement on the Map'); plt.xlabel('X position'); plt.ylabel('Y position'); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## MASAC route distribution


In [ ]:
x=evaluation['route'].value_counts()
ax=x.plot(kind='bar'); ax.set_title('MASAC Route Distribution'); ax.set_xlabel('Route'); ax.set_ylabel('Tasks'); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()
